In [ ]:
!pip install -U langchain langchain_community langchain_core
!pip install langchain-text-splitter
!pip install faiss-cpu
!pip install pypdf
!pip install bs4
!pip install langchain_google_genai


ERROR: Could not find a version that satisfies the requirement langchain-text-splitter (from versions: none)
ERROR: No matching distribution found for langchain-text-splitter


In [ ]:
!pip install langchain_chroma

In [ ]:
import getpass
import os
os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API key: ")




Enter your Google API key: ··········


In [ ]:
# documment datastructure what exactly is it ?
"""The Document data structure in Langchain is a container for text data and its associated metadata.
 It has two main parts: page_content which holds the text itself, and metadata which is a dictionary for storing extra information like the source or page number.
  This helps Langchain organize and understand the text it's working with."""



"The Document data structure in Langchain is a container for text data and its associated metadata.\n It has two main parts: page_content which holds the text itself, and metadata which is a dictionary for storing extra information like the source or page number.\n  This helps Langchain organize and understand the text it's working with."

In [ ]:
import bs4
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

In [ ]:

from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationChain

In [ ]:
llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash",temperature=0)


In [ ]:
news_path=input("Enter the news url for the summarizaion")

Enter the news url for the summarizaionhttps://www.bbc.com/news/articles/cx2jyv3jp01o


In [ ]:

loder=WebBaseLoader(web_path=news_path,
                    bs_kwargs=dict(
                        parse_only=bs4.SoupStrainer(id="main-content")
                    ))

In [ ]:
docs=loder.load()

In [ ]:
text_split=RecursiveCharacterTextSplitter(chunk_size=1000,
                                          chunk_overlap=200)
split=text_split.split_documents(docs)

In [ ]:
split[:3]

[Document(metadata={'source': 'https://www.bbc.com/news/articles/cx2jyv3jp01o'}, page_content='\'They hit so hard the house was shaking\': Iranians describe impact of US-Israel attacks1 hour agoShareSaveGhoncheh HabibiazadBBC News PersianShareSaveReutersIsrael and the US began attacking Iran on Saturday morningPeople in Iran have been describing their experiences as Israel and the US continue to bomb targets across the country.The attacks began on Saturday morning with the killing of Iran\'s Supreme Leader Ayatollah Ali Khamenei, shocking supporters and opponents. Since then, military and strategic sites have been attacked day and night. The authorities have blocked the internet, making it very difficult for people to communicate with the outside world. Despite the restrictions, the BBC has managed to speak some individuals. Their names have been changed to protect their identities.Hossein, in the city of Karaj, west of the capital Tehran, said there was a big blast near his home on Mo

In [ ]:
model_name="BAAI/bge-Base-en-v1.5"
model=HuggingFaceEmbeddings(model_name=model_name,
                                model_kwargs={"device":"cpu"},
                                encode_kwargs={"normalize_embeddings":True})

/tmp/ipython-input-1590/1255604079.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  model=HuggingFaceEmbeddings(model_name=model_name,
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
db=Chroma.from_documents(documents=split,
                         embedding=model)

In [ ]:
retriver=db.as_retriever() # its work is to find the most relevant parts of your documents that relate to your question

In [ ]:
memory = ConversationBufferMemory(return_messages=True)

/tmp/ipython-input-1590/3614923015.py:1: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(return_messages=True)


In [ ]:
prompt1 = PromptTemplate.from_template("""
You are a professional News Summarization AI Agent.

Extract Who, What, When, Where, Why, How.
Remove noise and ads.
Stay unbiased.

<context>
{context}
</context>

Return:

Headline:
Detailed Summary (10-15 lines):
Key Points:
Why It Matters:
Category:
""")

In [ ]:
chatbot_prompt = PromptTemplate.from_template("""
You are an engaging and intelligent News Chatbot.

Behavior Rules:
- Answer ONLY from the given context
- Do NOT hallucinate
- If answer not found → "I couldn't find this information."
- Keep responses conversational and concise

<context>
{context}
</context>

Conversation History:
{history}

Human: {input}
AI:
""")

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
# chain=({"context":retriver |format_docs ,"input": RunnablePassthrough()} # simply takes the input it receives and passes it along without changing it. It's useful for when you want to include the original input in a chain of operations
#        | prompt1
#        | llm
#        | StrOutputParser())
chain = (
    {"context": retriver, "input": RunnablePassthrough()}
    | prompt1
    | llm
    | StrOutputParser()
)

In [ ]:
!pip install langchain_classic

In [ ]:
from langchain_core.runnables import RunnablePassthrough
# Chain
chat_chain = (
    {
        "context": retriver | format_docs,
        "input": RunnablePassthrough(),
        "history": lambda x: memory.load_memory_variables({})["history"]
    }
    | chatbot_prompt
    | llm
    | StrOutputParser()
)

# Chat loop
while True:
    query = input("User: ")

    if query.lower() in ["exit", "quit"]:
        print("Bye take care 😊")
        break

    response = chat_chain.invoke(query)

    memory.save_context({"input": query}, {"output": response})

    print("Chatbot:", response)

KeyboardInterrupt: Interrupted by user

In [ ]:
query = "Summarize this news article"
result = chain.invoke(query)

print(result)

Headline: US-Israel Attacks Intensify in Iran Following Supreme Leader Khamenei's Death, Internet Cut Off

Detailed Summary:
US and Israeli forces have launched intensified attacks across Iran, beginning Saturday morning with the killing of Supreme Leader Ayatollah Ali Khamenei. Military and strategic sites have been bombed day and night, causing widespread fear and anger among the populace. Residents in cities like Karaj and Tehran report hearing numerous explosions, with one resident describing their house shaking from a blast on Monday. The Iranian regime has responded by cutting off internet access for the third time this year, severely hindering communication and further infuriating citizens. People are stocking up on groceries, while security forces patrol the streets, and bakeries and petrol stations are busy. The death of Khamenei is seen as widening the existing gap between pro- and anti-government factions, with some citizens expressing exhaustion and a desire for the entire 